**Purpose**

The purpose of this project is to develop a content-based movie recommendation system that prioritizes relevance based on movie characteristics while also taking into account movie quality and the assumed reliability of user ratings as well. Popularity is intentionally downweighted but not excluded entirely with knowledge that poppular movies has significantly more ratings than less known ones.

Unlike commercial systems that might optimize for engagement or exposure, this model has as goal to provide more balanced and transparent recommendations. Similarity is the largest factor and additional factors are used to refine the results. 

**Theoretical background**

Several concepts from machine learning are used in this project.

- *TF-IDF (Term Frequency–Inverse Document Frequency)* is used to convert text data, such as genres and tags, into numerical feature vectors. It gives higher importance to words that are frequent in a specific movie but less common across the dataset.

- *Cosine similarity* is used to measure how similar two movies are. It is defined as the cosine of the angle between two vectors and produces values between -1 and 1. A higher value means that the movies are more similar.

- *Min-max scaling* is used to normalize values to a range between 0 and 1. This is done by subtracting the minimum value and dividing by the difference between the maximum and minimum value.

- A *logarithmic transformation* is applied to reduce the effect of very active users. This helps to handle skewed distributions where a few users have rated many movies.

- The *weighted rating formula* combines the average rating and the number of ratings to balance quality and popularity. This formula is inspired by external sources and is not taken directly from the resources mentioned in this course.

**Method**

The recommendation system uses user-generated movie tags as the main source of content information, with genres used as a complementary feature. 

It was developed iteratively, starting with a simple content-based model and gradually incorporating additional features to improve it.
In the initial version, only movie genres were used as features. The genres were converted into a space-separated text representation and vectorized using TF-IDF. This allowed the model to represent each movie as a numerical feature vector. Cosine similarity was then used to identify movies with similar content[1]. However, this genre-based model produced overly broad recommendations. Since many movies share the exact same genre combination, a large number of movies received relatively high similarity scores. 

In an a second version, genres and tags were combined into a single text representation and vectorized using TF-IDF. This converts text into numerical feature vectors, where words that are comon within a movie description but less common across the dataset receive higher value. Cosine similarity was then used to identify movies with similar content.

In the final model, tags are still the main source for similarity, while genres are used as a complement when tags are missing. This is important because approximately 38% of the movies in the dataset do not have tags. The first version combined with the second version highlights the problem with relying solely on genre and tags.

To improve the recommendation quality, a reranking step was applied after the initial similarity search. Each candidate movie was assigned a final score based on three components:

- Similarity score (weight 0.7), used as the main ranking signal
- Normalized weighted rating (weight 0.2), used to reflect overall movie quality
- Log-transformed rating count (weight 0.1), used as a limited popularity signal

The final score was a weighted combination of these factors. Similarity had the strongest influence in order to preserve relevance to the input movie, while rating-based signals were included as secondary factors. 

The weighted rating was used to make movie quality estimates more robust than a simple mean rating. A weighted rating approach inspired by IMDb’s previously described method was used to balance movie quality against the number of ratings [2]. This reduces the impact of movies with very few votes, while allowing movies with many ratings to retain more of their observed average. In practice, this means that a movie with few high ratings is not allowed to outrank a movie with a slightly lower average but with many more ratings.

Before this the weighted rating for movies (i.e., the final rating assigned to each movie) was calculated and normalized, user ratings were also adjusted at the individual level. Since 13% of all ratings were produced by 1% of users, ratings were reweighted based on user activity. No users were removed from the dataset. Instead, ratings from highly active users were given lower weight, reducing their influence while preserving all available data. This was done to reduce the influence of very active users on the overall movie ratings. Normalization was made for the comparison with similarity and popularity.

The popularity term had a relatively small weight and log transformation to prevent highly rated blockbuster movies from dominating the ranking purely because they had accumulated many votes.

The weights assigned to the three final components, (0.7; 0.2; 0.1), were chosen manually and were not systematically optimized. A more rigorous approach would involve tuning these values[2].

**Data analysis (EDA)**

The exploratory data analysis revealed several important patterns..

- *A strong popularity bias was observed in the dataset*. Most movies have very few ratings, while a few popular movies have thousands of ratings. This makes average ratings unreliable for movies with very few votes. Threfor a weighted rating formula was used, combining the movie’s average rating with the overall dataset average and the number of ratings. This aimed to balance quality and popularity.

- *User activity is highly uneven*. A small group of very active users (“superusers”)    contributes a large share of all ratings (1% of users rates 13% of the movies). These   users also tend to give lower ratings on average compared to typical users: 
  
  Median rating by superusers: 3.50  
  Median rating by others: 4.00    

  Average rating by superusers: 3.22
  Average rating by others: 3.61**  
  
Based on the average rating, this seams to create an "activity bias" in the data. To reduce this effect, a logarithmic user weighting was done, where users with many ratings have less influence. In addition, users with very few ratings were removed to improve data reliability.

- *Tags more usefull than genres*. Analysis of genres and tags showed that user-generated tags provide more meaningful descriptions of movies compared to genres alone. While genres are limited and often shared across many movies (19 genres in total), tags capture more specific themes and context. However, not all movies contain tags (approx. 38% missing). Therefore, genres were kept as a complement feature.

Based on these findings, genres and tags were combined into a single text feature. This feature was then transformed into numerical vectors using TF-IDF, which allowed the model to compute similarity between movies using cosine similarity.

**discussion**

While the weighting strategy reduces the influence of highly active users, it may increase the impact of users with very few ratings. This can introduce noise, as their ratings may not reliably reflect consistent preferences. A further analysis of low-activity users and their rating behavior could help evaluate their impact on the model’s performance. 

**Why not KNN?*
I considered using K‑Nearest Neighbors (KNN) because it is simple, easy to understand, and works well on small datasets. However, I decided not to use KNN for the full MovieLens dataset. The full dataset is very large, and KNN does not scale well. It requires computing similarities between all users or items, which becomes too slow and uses too much memory on a personal computer. Because of this, KNN is not practical for the full dataset, even though it has clear advantages on smaller ones.

**Why TD-IDF?**
I chose to use a TF‑IDF based content‑based approach instead of K‑Nearest Neighbors. While KNN is simple and effective on small datasets, it does not scale well to the full MovieLens dataset, which contains over 86,000 movies.  TF‑IDF, on the other hand, is much more efficient and allows me to compute movie similarities without the heavy computational cost.

**Future improvements**  
In the final steps of the process collaborative filltering was considered based on the assumption and information gathered about how it is a more effective way of filterinng. It recommends movies based on trends and on which movies similar users enjoyed even from different genres. This could be a good way of improving the model.


[1] First genrebased-model discussed with Johan Challita.
[2] https://code.energy/write-code-rates-fairly/

